A.

In [60]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [58]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True,
    inferSchema=True
)

df_toko = spark.read.json(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json"
)

df_produk = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True,
    inferSchema=True
)

print("Transaksi:", df_transaksi.count())
print("Toko:", df_toko.count())
print("Produk:", df_produk.count())

Transaksi: 4000
Toko: 3
Produk: 15


B.

In [49]:
df_gabungan = df_transaksi.join(
    df_toko,
    on="cabang_id",
    how="inner"
).join(
    df_produk,
    on="menu_id",
    how="inner"
).withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


C.

In [50]:
penjualan_menu = df_gabungan.groupBy(
    "nama_cabang",
    "nama_menu"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy(
    "nama_cabang"
).orderBy(
    col("total_penjualan").desc()
)

top2_menu = penjualan_menu.withColumn(
    "peringkat",
    row_number().over(window_cabang)
).filter(
    col("peringkat") <= 2
)

top2_menu.show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



D. 

In [51]:
df_gabungan.createOrReplaceTempView("penjualan")

penjualan_barista = spark.sql("""
    SELECT
        kepala_barista,
        SUM(total_penjualan) AS total_penjualan
    FROM penjualan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



E. 

In [52]:
df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(
        "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
    )

F. 

In [53]:
ringkasan_eksekutif = df_gabungan.groupBy(
    "nama_cabang",
    "kota",
    "kepala_barista"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(
    col("total_penjualan").desc()
)

ringkasan_eksekutif.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



Berdasarkan hasil pengolahan data, cabang dengan kinerja terbaik adalah Kopi Nusantara Semarang dengan total penjualan sebesar 90.542.000. Cabang ini juga memiliki jumlah transaksi paling banyak, yaitu 1.393 transaksi. Sementara itu, cabang dengan kinerja terlemah adalah Kopi Nusantara Yogyakarta dengan total penjualan sebesar 84.645.000. Berdasarkan hasil bagian C, menu yang sebaiknya lebih diprioritaskan dalam promosi di cabang Yogyakarta adalah Menu-12 karena memiliki total penjualan tertinggi di cabang tersebut, yaitu sebesar 9.275.000. Menu-12 dapat dipromosikan dengan memberikan diskon, membuat paket menu, atau menampilkannya sebagai rekomendasi kepada pelanggan. Dengan memprioritaskan menu yang sudah memiliki penjualan tinggi, diharapkan pelanggan semakin tertarik untuk membeli menu tersebut. Hal ini juga dapat membantu meningkatkan total penjualan pada cabang Yogyakarta. Rekomendasi ini dibuat berdasarkan hasil pengolahan data transaksi dan perbandingan penjualan setiap cabang serta menu.

eksplorasi

In [61]:
hasil = df_gabungan.groupBy("kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

hasil.show()

+--------------+---------------+
| kategori_menu|total_penjualan|
+--------------+---------------+
|Makanan Ringan|      142757000|
|      Non-Kopi|       55322000|
|          Kopi|       64246000|
+--------------+---------------+



In [62]:
hasil = df_gabungan.groupBy("kota").agg(
    avg("total_penjualan").alias("rata_rata_penjualan")
)

hasil.show()

+----------+-------------------+
|      kota|rata_rata_penjualan|
+----------+-------------------+
|  Magelang|  66926.26728110599|
|  Semarang|   64997.8463747308|
|Yogyakarta|  64862.06896551724|
+----------+-------------------+



In [63]:
hasil = df_gabungan.groupBy("nama_cabang").count()

hasil.show()

+--------------------+-----+
|         nama_cabang|count|
+--------------------+-----+
|Kopi Nusantara Yo...| 1305|
|Kopi Nusantara Ma...| 1302|
|Kopi Nusantara Se...| 1393|
+--------------------+-----+



In [64]:
hasil = df_gabungan.orderBy(
    col("total_penjualan").desc()
).limit(5)

hasil.show()

+-------+---------+------+---+--------------+--------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|    kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+--------+--------------------+---------+--------------+-----+---------------+
|     12|        1|  KN46|  4|         Nadia|Magelang|Kopi Nusantara Ma...|  Menu-12|Makanan Ringan|35000|         140000|
|     12|        3| KN133|  4|         Nadia|Semarang|Kopi Nusantara Se...|  Menu-12|Makanan Ringan|35000|         140000|
|      1|        3|  KN79|  4|         Nadia|Semarang|Kopi Nusantara Se...|   Menu-1|Makanan Ringan|35000|         140000|
|      9|        3|   KN8|  4|         Nadia|Semarang|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|35000|         140000|
|      9|        3|  KN98|  4|         Nadia|Semarang|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|35000|         140000|
+-------+-------

In [67]:
hasil = df_gabungan.filter(
    col("qty") >= 2
)

hasil.show()

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|
|     14|        2|   KN5|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-14|          Kopi|18000|          54000|
